## Decision Tree- Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (8,4)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 1000)
#pd.set_option('display.max_columns', 1000)
#pd.set_option('display.width', 1000)

In [ ]:
df=pd.read_csv("car_dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().any()

In [ ]:
df.describe().T

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, vmin=-1, vmax=1, cmap="coolwarm")
plt.show()

In [ ]:
df.head()

In [ ]:
df["vehicle_age"]=2022-df.Year

In [ ]:
df.head()

In [ ]:
df.select_dtypes("object").head()

In [ ]:
for i in df.select_dtypes("object"):

    print(i, len(df[i].value_counts()))

# we see how many unique observations the categorical features consist of.
# The number of unique categorical observations of car_name is very high compared to
# the total number of observations in our data.

In [ ]:
df.Car_Name.value_counts()

In [ ]:
df.drop(columns=["Car_Name", "Year"], inplace=True)

# Although I have 301 observations, there are 98 unique car/engine brands in car_name.
# Converting to numeric data by applying ordinal encoder to this feature will not be a meaningful course of action.
# Because the DT model tends to place more emphasis on high cardinality categorical features
# (features that contain too many unique categorical observations).
# This causes these featurs to come to the fore in feature importance.

# There are also 1-2 of most brands and models. Naturally, we are removing this feature as
# it will not contribute to education.

# Since we have converted the Year feature, which is the year of vehicle production,
# to the Age feature, we are removing the Year feature along with the car_name.

In [ ]:
df.head()

In [ ]:
sns.histplot(df.Selling_Price, bins=50, kde=True);

# tree based models are very robust against outliers. However, this is not the case for small regression data.
# Since outlier values in small regression data can seriously increase or decrease the average prices on fallen leaves,
# it is important to reduce these outlier observations, which are insufficient for training, in tree-based models.

# However, as always, we should compare the scores both with the outliers and
# by reducing the outliers and see the differences.

In [ ]:
sns.boxplot(df.Selling_Price);

In [ ]:
df2 = df.copy()

## Train test split

In [ ]:
X=df.drop("Selling_Price", axis=1)
y=df.Selling_Price

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)

print("Train features shape : ", X_train.shape)
print("Train target shape   : ", y_train.shape)
print("Test features shape  : ", X_test.shape)
print("Test target shape    : ", y_test.shape)

## Modeling with Pipeline

In [ ]:
cat = X_train.select_dtypes("object").columns
cat

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder


ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

column_trans = make_column_transformer((ord_enc, cat),
                                       remainder='passthrough',
                                       verbose_feature_names_out=False).set_output(transform="pandas")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(random_state=101))]

pipe_model = Pipeline(steps=operations)

pipe_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

In [ ]:
def train_val(model, X_train, y_train, X_test, y_test):

    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    scores = {"train": {"R2" : r2_score(y_train, y_train_pred),
    "mae" : mean_absolute_error(y_train, y_train_pred),
    "mse" : mean_squared_error(y_train, y_train_pred),
    "rmse" : mean_squared_error(y_train, y_train_pred)**0.5, #"rmse" : mean_squared_error(y_train, y_train_pred, squared=False)
    "mape" :mean_absolute_percentage_error(y_train, y_train_pred)},

    "test": {"R2" : r2_score(y_test, y_pred),
    "mae" : mean_absolute_error(y_test, y_pred),
    "mse" : mean_squared_error(y_test, y_pred),
    "rmse" : mean_squared_error(y_test, y_pred)**0.5, #"rmse" : mean_squared_error(y_test, y_pred, squared=False)
    "mape" :mean_absolute_percentage_error(y_test, y_pred)}}

    return pd.DataFrame(scores)

In [ ]:
train_val(pipe_model, X_train, y_train, X_test, y_test)

# We got our insight that was overfitting. We will make our final decision on the CV.

In [ ]:
pd.options.display.float_format = '{:.3f}'.format
train_val(pipe_model, X_train, y_train, X_test, y_test)

In [ ]:
from sklearn.model_selection import cross_validate

operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(random_state=101))]
model = Pipeline(steps=operations)

scores = cross_validate(model,
                        X_train,
                        y_train,
                        scoring=['r2',
                                 'neg_mean_absolute_error',
                                 'neg_mean_squared_error',
                                 'neg_root_mean_squared_error',
                                 'neg_mean_absolute_percentage_error'],
                        cv =10,
                        return_train_score=True)

df_scores = pd.DataFrame(scores)
df_scores.mean()[2:]

# There is over fitting

## Removing Outliers

In [ ]:
from yellowbrick.regressor import PredictionError
from yellowbrick.features import RadViz

visualizer = RadViz(size=(720, 600))

model = pipe_model
visualizer = PredictionError(model)
visualizer.fit(X_train, y_train) # Fit the training data to the visualizer
visualizer.score(X_test, y_test) # Evaluate the model on the test data
visualizer.show();

# We found that prices greater than 10 worsened the forecasts.

In [ ]:
len(df[df.Selling_Price > 10])

# There are 28 observations over 10 in the data. And since the whole data rate is 9%, we drop it.

In [ ]:
28/301

In [ ]:
df_new = df[df.Selling_Price < 10]
df_new.head()

# We filtered out prices greater than 10

In [ ]:
X = df_new.drop(columns="Selling_Price")
y = df_new.Selling_Price

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)

In [ ]:
operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(random_state=101))]

pipe_model = Pipeline(steps=operations)

pipe_model.fit(X_train, y_train)

In [ ]:
train_val(pipe_model, X_train, y_train, X_test, y_test)

# After filtering, the scores of our test data got better, but the overfiting still could not be fixed.

In [ ]:
visualizer = RadViz(size=(500, 1000))

model = pipe_model
visualizer = PredictionError(model)
visualizer.fit(X_train, y_train) # Fit the training data to the visualizer
visualizer.score(X_test, y_test) # Evaluate the model on the test data
visualizer.show();

# We see here that the scores got better after filtering.

## Visualizing trees

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
features = pipe_model["OrdinalEncoder"].get_feature_names_out()

features

# We keep in mind that the feature order has changed since the make_column_transform function is used in the pipeline.

In [ ]:
pipe_model["OrdinalEncoder"].fit_transform(X_train).head()

In [ ]:
X_train.head(1)

In [ ]:
plt.figure(figsize=(12,8), dpi=150)
plot_tree(pipe_model["DT_model"], filled=True, feature_names=features.tolist()); #feature_names=X.columns

# Branching in regression problems of tree-based models is much more than in classification problems.

In [ ]:
def report_model(model):
    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)
    print('\n')
    print(train_val(model, X_train, y_train, X_test, y_test))
    print('\n')
    plt.figure(figsize=(12,8),dpi=100)
    plot_tree(model["DT_model"],filled=True, feature_names=features.tolist(), fontsize=8); #feature_names=X.columns

In [ ]:
operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(criterion='squared_error',
                                                 max_depth=3,
                                                 random_state=101))]

pruned_tree = Pipeline(steps=operations)
pruned_tree.fit(X_train,y_train)

report_model(pruned_tree)

# The only difference from clasification is criterion='squared_error'. If you remember,
# this hyper_parameter was gini or entropy in clasification.
# In regression, criterion= "squared_error", "friedman_mse", "absolute_error", "poisson"
# can take one of the values. But usually this value is used by default because the model gets
# the best results in "squared_error".

# Squared error represents mean squared error. A leaf's error of 0 indicates that
# all the cars falling on that leaf are of the same price, while a value close to 0 indicates
# that the prices are very close to each other. A high error value indicates that the prices of
# that leaf are very close to each other. indicates that irrelevant cars have crashed.

# sampales shows the total number of observations in the corresponding leaf.

# value shows the average price of the observations in the relevant sheet.

# The value of the leaf on whichever leaf we predict falls on, becomes the value predicted by the model of the car.

## GridSearch

In [ ]:
DecisionTreeRegressor?

In [ ]:
from sklearn.model_selection import GridSearchCV, validation_curve
from yellowbrick.model_selection import ValidationCurve
from ipywidgets import interact
import ipywidgets as widgets

In [ ]:
# Visualization function
def validation_curve(criterion):
  operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(criterion=criterion, random_state=101))]

  pipe_model = Pipeline(steps=operations)

  viz = ValidationCurve(
      pipe_model, param_name="DT_model__ccp_alpha", param_range=np.linspace(0, 0.1, 25),
      logx=False, cv=10, scoring="neg_root_mean_squared_error", n_jobs=-1,
  )

  viz.fit(X_train, y_train)
  viz.show()

# Create a dropdown widget for criterion selection
criterion_widget = widgets.Dropdown(options=['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
                                    value='absolute_error',
                                    description='Criterion:',
                                    disabled=False)

# Use interact to create the interactive widget
interact(validation_curve, criterion=criterion_widget)


In [ ]:
param_grid = {"DT_model__criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
              "DT_model__ccp_alpha": [.04, .043, .05]}

In [ ]:
operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(random_state=101))]

pipe_model = Pipeline(steps=operations)
grid_model = GridSearchCV(estimator=pipe_model,
                          param_grid=param_grid,
                          scoring='neg_root_mean_squared_error',
                          cv=10,
                          n_jobs=-1,
                          return_train_score=True)

In [ ]:
grid_model.fit(X_train,y_train)

In [ ]:
grid_model.best_estimator_

In [ ]:
pd.DataFrame(grid_model.cv_results_).loc[grid_model.best_index_, ["mean_test_score", "mean_train_score"]]
# overfitting problem is solved

In [ ]:
train_val(grid_model, X_train, y_train, X_test, y_test)

# We have seen that the test data scores and the validation set scores from the CV are consistent.

In [ ]:
from sklearn.model_selection import cross_validate, cross_val_score

operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(ccp_alpha=0.043,
                                                 criterion='absolute_error',
                                                 random_state=101))]

model = Pipeline(steps=operations)


scores = cross_validate(model,
                        X_train,
                        y_train,
                        scoring=['r2',
                                 'neg_mean_absolute_error',
                                 'neg_mean_squared_error',
                                 'neg_root_mean_squared_error',
                                 'neg_mean_absolute_percentage_error'],
                        cv = 10,
                        return_train_score=True)
df_scores = pd.DataFrame(scores, index=range(1,11))
df_scores.iloc[:,2:]

# We can make a CV to see all metrics.
# When we look at all iterations in the CV, we can see that some scores are very different.
# This is expected for very small data.

In [ ]:
df_scores.mean()[2:]

In [ ]:
train_val(grid_model, X_train, y_train, X_test, y_test)

## Feature İmportance

In [ ]:
operations = [("OrdinalEncoder", column_trans),
              ("DT_model", DecisionTreeRegressor(ccp_alpha=0.043,
                                                 criterion='absolute_error',
                                                 random_state=101))]
pipe_model = Pipeline(steps=operations)
pipe_model.fit(X_train, y_train)

# We do not forget that we should use the best hyper parameters we found after gridsearch for feature importance.

In [ ]:
X_train.head(1)

# Since make_colum_transform is used in the pipeline, we do not use feture names here.

In [ ]:
features

In [ ]:
pipe_model["OrdinalEncoder"].fit_transform(X_train).head()

# After make_colum_transform, we use the features list that shows the feature names of the new df in order.

In [ ]:
df_f_i = pd.DataFrame(data = pipe_model["DT_model"].feature_importances_,
                      index=features,
                      columns=["Feature Importance"])

df_f_i = df_f_i.sort_values("Feature Importance", ascending=False)

df_f_i

# DT couldn't get any insight from the other features except for the 2 features
# Because DT always concentrates on the best features,
# in later iterations we can drop the dominant feature and
# try to see if the dominance comes from high cardinality.

In [ ]:
ax = sns.barplot(x = df_f_i.index, y = 'Feature Importance', data = df_f_i)
ax.bar_label(ax.containers[0],fmt="%.3f")
plt.xticks(rotation = 90)
plt.tight_layout()

## Compare real and predicted result

In [ ]:
y_pred = grid_model.predict(X_test)
my_dict = { 'Actual': y_test, 'Pred': y_pred, 'Residual': y_test-y_pred }
compare = pd.DataFrame(my_dict)

# we compare the actual, predicted values ​​with the residuals to see if the model can make the predictions successfully.

In [ ]:
comp_sample = compare.sample(20)
comp_sample

In [ ]:
comp_sample.plot(kind='bar',figsize=(9,5))
plt.show()

## Final Model

In [ ]:
X=df_new.drop("Selling_Price", axis=1)
y=df_new.Selling_Price

In [ ]:
from sklearn.pipeline import Pipeline

operations = [("Ordinalcoder", column_trans),
              ("DT_model", DecisionTreeRegressor(ccp_alpha=0.043,
                                                 criterion='absolute_error',
                                                 random_state=101))]

pipe_model = Pipeline(steps=operations)

pipe_model.fit(X, y)

# we re-build our final model for all data with the best hyper parameters.

## Prediction

In [ ]:
X.describe()

In [ ]:
samples = {"Present_Price": [7, 9.5],
           "Kms_Driven": [25000, 44000],
           "Fuel_Type":["Petrol", "Diesel"],
           'Seller_Type':['Dealer', 'Individual'],
           'Transmission': ['Manual', 'Automatic'],
           "Owner":[0,1],
           "vehicle_age":[1, 9]}

In [ ]:
df_samples = pd.DataFrame(samples)
df_samples

In [ ]:
pipe_model.predict(df_samples)

In [ ]:
import nbformat

file = 'Car Dataset-Decision Tree Regression.ipynb'
nb = nbformat.read(file, as_version=nbformat.NO_CONVERT)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']
    nbformat.write(nb, file)
    print(f"Fixed and saved: {file}")